# Explorer un enregistrement, une fenêtre et son négatif apparié

Suivre une fenêtre à travers la chaîne : l'enregistrement, sa grille de fenêtres, le module séquentiel (en amont, en parallèle, en aval), le négatif apparié, les embeddings et le prototype différentiel.

- **Lecture seule** : la base est ouverte en lecture, l'audio des disques externes est lu, jamais écrit.
- **Ne pas committer les sorties** : les lecteurs audio embarquent le son des enregistrements (données de l'ONF et de Biophonia). Avant un commit : *Clear All Outputs* (`tests/test_notebooks.py` le vérifie).
- Tout se règle dans la cellule **Réglages**, puis *Run All*.
- Noyau : l'environnement du dépôt (`.venv`), après `uv sync --group notebook`.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from blanci import explore as ex
from blanci import explore_plots as ep
from blanci.audio import resample
from blanci.config import config_path, load_config
from blanci.sequential import (
    persistence_features,
    recording_onsets,
    rhythm_features,
    upstream_from_cfg,
    window_rhythm,
)

# Racine du dépôt : les chemins de la config y sont relatifs (le notebook tourne dans notebooks/).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 200)

## Réglages

In [ ]:
CONFIG = ROOT / "config" / "local.yaml"  # paths.raw = disque externe ; config par défaut sinon
ENREGISTREMENT = None  # recording_id ; None = le RANG-ième enregistrement à annotation positive
RANG = 0  # 0 = le plus annoté
FENETRE_S = None  # début (s) de la fenêtre choisie ; None = la première annotée positive
CANAL = None  # 0 (micro à 6 dB), 1 (18 dB) ou "mean" ; None = audio.channel de la config
CHEVAUCHEMENT = None  # 0 à 0,99 ; None = grille de la config (w3 : 3 s, pas de 1,5 s)
AMONT = None  # fonctionnalités en amont, ex. ["bandpass", "denoise", "notes"] ; None = config
STRATEGIE = None  # nearest, same_day, other_day ou mixed ; None = benchmark.pairing
NEGATIF = 0  # rang du négatif apparié (0 = le plus proche de la fenêtre)
STOCK = None  # stock d'embeddings (data/embeddings/<stock>) ; None = le premier trouvé
ENCODEUR_DIRECT = None  # ex. "perch_v2" : sans stock, encode les deux fenêtres à la volée
SEUIL_DETECTEUR = 0.5  # BlanciNet : fenêtre positive au-dessus (persistance, en aval)

cfg = load_config(CONFIG if CONFIG.exists() else None)
if CANAL is not None:
    cfg["audio"]["channel"] = CANAL
if STRATEGIE is not None:
    cfg["benchmark"]["pairing"] = STRATEGIE
con = ex.open_readonly(config_path(cfg, "db"))
print("audio :", config_path(cfg, "raw"), "| canal :", cfg["audio"]["channel"])
print("négatifs appariés :", cfg["benchmark"]["pairing"])

## 1. Choisir un enregistrement

Les enregistrements à annotation positive d'abord, les plus annotés en tête, avec les détections BlanciNet (nombre, score max). Pour en prendre un autre : `ENREGISTREMENT` ou `RANG` dans **Réglages** ; n'importe quel enregistrement de la base convient (`overview` les contient tous).

In [ ]:
overview = ex.recordings_overview(con, cfg)
positives = overview[overview["n_positive"] > 0]
counts = ("n_positive", "n_labelled", "n_negative")
detector_columns = [c for c in overview.columns if c.startswith(("n_", "max_")) and c not in counts]
print(f"{len(overview)} enregistrements, dont {len(positives)} à annotation positive")
shown = ["recording_id", "point", "local", "n_positive", "n_negative", *detector_columns]
positives[shown].head(20)

In [ ]:
rid = ENREGISTREMENT or positives["recording_id"].iloc[RANG]
rec = overview.loc[overview["recording_id"] == rid].iloc[0]
rec.drop("qc_flags").to_frame("enregistrement")

## 2. Écouter l'enregistrement

Spectrogramme de l'enregistrement entier (traits cyan : débuts de notes détectés ; pointillés : bande d'A. blanci). Dessous, une piste par sorte de fenêtre : annotées positives, annotées négatives, faux négatifs suspects (non annotées, encadrées de positives, DECISIONS n° 102) ; puis les scores BlanciNet (détections du prestataire, ≥ 0,1).

In [ ]:
wav, sr = ex.read_recording(cfg, rec["path"])
windows = ex.recording_windows(con, cfg, rid, overlap=CHEVAUCHEMENT)
onsets = recording_onsets(wav, sr, cfg["signal"])
dur = float(windows["dur_s"].iloc[0])
print(f"{len(wav) / sr:.0f} s à {sr} Hz ; {len(windows)} fenêtres de {dur:g} s")
print(
    f"{len(onsets)} débuts de notes détectés (signal.onset_k_mad = {cfg['signal']['onset_k_mad']})"
)
title = f"{rid} — {rec['point']} — {rec['local']}"
ep.plot_recording(wav, sr, windows, cfg, onsets=onsets, title=title);

In [ ]:
ep.listen(wav, sr)

## 3. Les fenêtres

Chaque fenêtre de la grille : label (annotation transférée à la grille), chevauchement d'une annotation positive, distance à la plus proche, faux négatif suspect, score BlanciNet, et les valeurs des quatre portes du module séquentiel en amont — énergie et contraste en bande (sur le son de la fenêtre), notes et intervalles d'A. blanci (sur les débuts de notes de l'enregistrement).

In [ ]:
values = ex.window_indices(wav, sr, windows, cfg, onsets)
table = windows.join(values)
table.drop(columns=["window_id", "recording_id"]).round(2)

Les portes le long de l'enregistrement. Trait plein : seuil d'une porte activée (`AMONT` ou config) ; pointillé : porte désactivée.

In [ ]:
upstream = upstream_from_cfg(cfg, only=AMONT)
ep.plot_indices(windows, values, cfg, upstream);

### Écouter des fenêtres

`A_ECOUTER` : débuts (s) des fenêtres à écouter. Par défaut, les trois premières annotées positives et les trois fenêtres libres (sans annotation positive) les plus proches d'une annotation — celles que `nearest` tire en premier.

In [ ]:
free = windows[~windows["overlaps_positive"]].sort_values(["distance_to_positive_s", "offset_s"])
A_ECOUTER = sorted(
    windows.loc[windows["y"] == 1, "offset_s"].head(3).tolist() + free["offset_s"].head(3).tolist()
)
detectors = [c for c in windows.columns if c not in ex.WINDOW_COLUMNS]
for offset in A_ECOUTER:
    row = table.loc[table["offset_s"] == offset].iloc[0]
    label = row["label"] if isinstance(row["label"], str) else "non annotée"
    scores = ", ".join(f"{c} {row[c]:.2f}" for c in detectors if pd.notna(row[c]))
    contrast = f"contraste {row['band_contrast']:.1f} dB"
    print(f"{offset:6.1f} s — {label} — {contrast} — {scores or 'rien détecté'}")
    display(ep.listen(ex.cut(wav, sr, offset, dur), sr))
ep.spectrogram_grid([(f"{o:g} s", ex.cut(wav, sr, o, dur)) for o in A_ECOUTER], sr, cfg);

## 4. La fenêtre choisie

`FENETRE_S` dans **Réglages** (début en s) ; par défaut la première fenêtre annotée positive. Sous le spectrogramme : l'enveloppe en bande, le seuil de détection des notes (médiane + `signal.onset_k_mad` × MAD) et les débuts de notes retenus (durée compatible avec une note d'A. blanci), détectés dans la fenêtre seule.

In [ ]:
if FENETRE_S is None:
    annotated = windows.loc[windows["y"] == 1, "offset_s"]
    FENETRE_S = float(
        annotated.iloc[0] if len(annotated) else windows["offset_s"].iloc[len(windows) // 2]
    )
chosen = table.loc[(table["offset_s"] - FENETRE_S).abs().idxmin()]
segment = ex.cut(wav, sr, chosen["offset_s"], dur)
name = chosen["label"] if isinstance(chosen["label"], str) else "non annotée"
display(chosen.drop(["window_id", "recording_id"]).to_frame("fenêtre choisie"))
ep.plot_window(
    segment, sr, cfg, title=f"{chosen['offset_s']:g}–{chosen['offset_s'] + dur:g} s ({name})"
);

In [ ]:
ep.listen(segment, sr)

## 5. Le module séquentiel

`sequential.position` dit où il agit : en amont (avant l'encodeur : transformations du son et portes), en parallèle (rythme fusionné avec le score de la tête), en aval (persistance, après la tête).

In [ ]:
features = [*upstream.transforms, *upstream.gates]
print("position :", cfg["sequential"]["position"])
print("en amont, vues ici :", features or "aucune (AMONT, ou sequential.upstream.*.enabled)")

### 5.1 En amont : transformations

Le son que l'encodeur recevrait, étape par étape, même échelle de couleurs. Sans transformation activée, on montre quand même passe-bande et débruitage, pour voir leur effet.

In [ ]:
shown = upstream if upstream.transforms else upstream_from_cfg(cfg, only=["bandpass", "denoise"])
stages = ex.upstream_stages(segment, sr, shown)
ep.plot_stages(stages, sr, cfg);

In [ ]:
for stage_name, stage in stages:
    print(stage_name)
    display(ep.listen(stage, sr))

### 5.2 En amont : portes

Une fenêtre qui ne passe pas les portes activées n'est pas encodée ; son score est le plus bas (un positif arrêté est un positif manqué).

In [ ]:
passed = upstream.passes(values.loc[[chosen.name]])[0]
kept = upstream.passes(values)
lost = int(((windows["y"] == 1) & ~kept).sum())
print(f"portes activées : {upstream.gates or 'aucune'} ({upstream.combine})")
print(f"fenêtre choisie : {'encodée' if passed else 'arrêtée'}")
print(f"enregistrement : {kept.sum()} fenêtres encodées sur {len(kept)}")
print(f"positives annotées arrêtées : {lost}")
ex.gate_report(chosen, cfg, upstream)

### 5.3 En parallèle : rythme

Descripteurs de rythme de la fenêtre (débuts de notes qui y tombent) et de l'enregistrement entier ; ceux de `sequential.columns` sont fusionnés avec le score de la tête.

In [ ]:
ioi = tuple(cfg["signal"]["ioi_range_s"])
rhythm = window_rhythm(windows, {rid: onsets}, ioi)
print("colonnes fusionnées :", cfg["sequential"]["columns"])
pd.DataFrame(
    {
        "fenêtre": rhythm.loc[chosen.name],
        "enregistrement": pd.Series(rhythm_features(onsets, len(wav) / sr, ioi)),
    }
)

### 5.4 En aval : persistance

Sur les scores de toutes les fenêtres de l'enregistrement. Sans tête entraînée : ceux de BlanciNet (0 là où il n'a rien détecté), positifs au-dessus de `SEUIL_DETECTEUR`. Les détections BlanciNet sont sur une grille de 3 s jointives, reportées ici sur la grille de fenêtres.

In [ ]:
if detectors:
    scores = windows[detectors[0]].fillna(0.0).to_numpy()
    offsets = windows["offset_s"].to_numpy()
    radius = cfg["sequential"]["gap_radius_s"]
    persistence = persistence_features(scores, SEUIL_DETECTEUR, offsets, radius)
    display(pd.Series(persistence).to_frame(f"{detectors[0]} ≥ {SEUIL_DETECTEUR}"))
else:
    print("aucun score de détecteur pour cet enregistrement")

## 6. Le négatif apparié

Les négatifs appariés sont *présumés* : ni annotés positifs, ni écoutés. Ils sont tirés par `paired_negatives` comme au benchmark, selon `benchmark.pairing` (ou `STRATEGIE`), toujours du même micro. Avec `nearest` : les fenêtres les plus proches d'une annotation positive dans l'enregistrement lui-même (sans la chevaucher, sans être encadrées de positives), puis le même jour, puis un autre jour. Rangés ici du plus proche au plus loin de la fenêtre choisie ; `NEGATIF` choisit le rang.

Une fenêtre non annotée n'est pas une fenêtre écoutée : le score BlanciNet du négatif dit si le détecteur du prestataire y entend quelque chose. `02_negatifs_apparies` mesure ce que contiennent ces négatifs.

In [ ]:
if rec["n_positive"] > 0:
    negatives = ex.paired_for_recording(con, cfg, rid)
else:  # sans annotation positive : les négatifs qu'aurait la fenêtre si elle était positive
    hypothetical = pd.DataFrame(
        [{"recording_id": rid, "offset_s": chosen["offset_s"], "dur_s": dur}]
    )
    negatives = ex.paired_for_recording(con, cfg, rid, positive_windows=hypothetical)
negatives = ex.nearest_first(negatives, chosen["offset_s"])
negatives = negatives.join(ex.detector_scores(con, negatives))
negatives.drop(columns=["window_id", "label", "y", "path"])

In [ ]:
negative = negatives.iloc[NEGATIF]
if negative["recording_id"] == rid:
    neg_wav, neg_onsets = wav, onsets
else:
    neg_wav, neg_sr = ex.read_recording(cfg, negative["path"])
    neg_wav = resample(neg_wav, neg_sr, sr)
    neg_onsets = recording_onsets(neg_wav, sr, cfg["signal"])
neg_segment = ex.cut(neg_wav, sr, negative["offset_s"], dur)
placed = negatives.iloc[[NEGATIF]].assign(dur_s=dur)
neg_values = ex.window_indices(neg_wav, sr, placed, cfg, neg_onsets).iloc[0]
where = f"{negative['local']}, à {negative['offset_s']:g} s"
print(f"négatif n° {NEGATIF} : {negative['pairing']}, {where}")
title = "négatifs appariés tirés dans l'enregistrement, fenêtre choisie"
ep.plot_recording(
    wav,
    sr,
    windows,
    cfg,
    onsets=onsets,
    negatives=negatives,
    chosen=chosen["offset_s"],
    title=title,
);

In [ ]:
ep.plot_window(neg_segment, sr, cfg, title=f"négatif apparié ({negative['pairing']})");

In [ ]:
ep.listen(neg_segment, sr)

### Comparer la fenêtre et son négatif

Même échelle de couleurs. La différence en dB montre ce que la fenêtre a en plus (rouge) ou en moins (bleu) que son négatif : ce qui reste quand on retranche le fond partagé. Lecteurs au niveau réel (non normalisés) pour comparer à l'oreille.

In [ ]:
ep.plot_pair(segment, neg_segment, sr, cfg, titles=("fenêtre", f"négatif ({negative['pairing']})"));

In [ ]:
comparison = pd.DataFrame({"fenêtre": values.loc[chosen.name], "négatif": neg_values})
comparison["écart"] = comparison["fenêtre"] - comparison["négatif"]
display(comparison.round(2))
display(ep.listen(segment, sr, normalize=False), ep.listen(neg_segment, sr, normalize=False))

## 7. Embeddings

Un stock d'embeddings existe (`data/embeddings/<stock>`, écrit par `blanci embed`) : on y lit les fenêtres les plus proches de la fenêtre et du négatif. Sinon, `ENCODEUR_DIRECT` (ex. `"perch_v2"`) encode ces deux fenêtres à la volée (chargement du modèle : une minute au premier appel). Sinon, rien.

In [ ]:
stocks = ex.embedding_stocks(cfg)
stock = STOCK or (stocks[0] if stocks else None)
print("stocks présents :", stocks or "aucun")
e_pos = e_neg = None
if stock:
    window_s = ex.stock_window_s(con, stock, dur)
    meta, emb = ex.recording_embeddings(cfg, stock, rec)
    neg_rec = overview.loc[overview["recording_id"] == negative["recording_id"]].iloc[0]
    neg_meta, neg_emb = ex.recording_embeddings(cfg, stock, neg_rec)
    if len(meta) and len(neg_meta):
        row_pos, e_pos = ex.nearest_embedding(meta, emb, chosen["center_s"], window_s)
        neg_center = negative["offset_s"] + dur / 2
        row_neg, e_neg = ex.nearest_embedding(neg_meta, neg_emb, neg_center, window_s)
        offsets = f"{row_pos['offset_s']:g} s et {row_neg['offset_s']:g} s"
        print(f"{stock} : fenêtres de {window_s:g} s à {offsets}")
    else:
        print(f"{stock} : l'enregistrement ou son négatif n'est pas encodé")
elif ENCODEUR_DIRECT:
    from blanci.encoders import get_encoder

    encoder = get_encoder(ENCODEUR_DIRECT, cfg)
    half = encoder.window_s / 2
    batch = np.stack(
        [
            ex.cut(wav, sr, chosen["center_s"] - half, encoder.window_s),
            ex.cut(neg_wav, sr, negative["offset_s"] + dur / 2 - half, encoder.window_s),
        ]
    )
    e_pos, e_neg = encoder.embed(batch, sr)
    print(f"{ENCODEUR_DIRECT} : fenêtres de {encoder.window_s:g} s, dimension {encoder.dim}")
else:
    print("Aucun embedding : `blanci embed` d'abord, ou ENCODEUR_DIRECT dans Réglages.")

In [ ]:
if e_pos is not None:
    pair = ex.compare_embeddings(e_pos, e_neg)
    print(f"cosinus fenêtre–négatif : {pair['cosine']:.3f}")
    print(f"normes : {pair['norm_pos']:.2f} (fenêtre), {pair['norm_neg']:.2f} (négatif)")
    ep.plot_embeddings({"fenêtre": e_pos, "négatif": e_neg})

## 8. Le prototype différentiel

La tête `prototype` apprend w = μ₊ − μ₋ (centroïdes des embeddings normalisés des positifs et de leurs négatifs appariés) et score une fenêtre par ê·w + b, seuil 0 au milieu des centroïdes. Retrancher μ₋ retire ce que positifs et négatifs ont en commun : le fond sonore du micro, de l'heure, de la saison. Trois vues de cet effet :

1. **dans le son** (toujours disponible) : on retranche à la fenêtre le spectre médian de son négatif apparié ;
2. **sur la paire d'embeddings** : w = ê(fenêtre) − ê(négatif), tracé en section 7 ;
3. **sur tout le stock** (s'il existe) : w et b appris comme la tête, scores le long de l'enregistrement, contre le prototype simple (cosinus à μ₊, sans négatifs).

In [ ]:
cleaned = ex.subtract_background(segment, neg_segment, sr)
stages = [
    ("fenêtre", segment),
    ("négatif apparié", neg_segment),
    ("fenêtre − fond du négatif", cleaned),
]
ep.plot_stages(stages, sr, cfg);

In [ ]:
ep.listen(cleaned, sr)

In [ ]:
if stock and e_pos is not None:
    prototypes = ex.corpus_prototypes(con, cfg, stock)
    print(f"{prototypes['n_pos']} positifs, {prototypes['n_neg']} négatifs")
    cosine = prototypes["cosine_centroids"]
    print(f"cosinus entre μ₊ et μ₋ : {cosine:.3f} (proche de 1 : l'embedding code surtout le fond)")
    along = ex.prototype_scores(emb, prototypes)
    centers = meta["offset_s"].to_numpy() + window_s / 2
    ep.plot_scores_along(
        centers, along, windows, title=f"{stock} : prototypes le long de l'enregistrement"
    )
    display(
        ex.prototype_scores(np.stack([e_pos, e_neg]), prototypes).set_axis(["fenêtre", "négatif"])
    )
    w = prototypes["w"] / np.linalg.norm(prototypes["w"])
    d = pair["difference"] / max(np.linalg.norm(pair["difference"]), 1e-12)
    print(f"cosinus entre w (corpus) et la différence de la paire : {float(w @ d):.3f}")
else:
    print("Sans stock d'embeddings, pas de prototype appris sur le corpus.")

## 9. Et ensuite

- `02_negatifs_apparies.ipynb` : ce que contiennent les négatifs appariés, stratégie par stratégie, et une feuille d'écoute.
- `03_module_sequentiel.ipynb` : régler la détection des notes et les seuils des portes.